# A7: Multi-Model Evaluation and Cost Scaling

This notebook contains the Part 1 benchmark and scoring work, followed by a Part 2 cost and capacity simulation. It uses the same 15 models and completed benchmark data; the simulation does not download or run any models. Part 3 routing strategies are documented in system_design.md and are outside this notebook's scope.

The Part 1 run contains 375 responses (15 models × 25 prompts). The records include generation latency, output tokens per second, input/output token counts, and reviewed 1–10 scores. Part 2 assumes five requests per DAU per day, estimates self-hosted capacity from mean generation latency, and calculates Hugging Face Inference Providers usage costs from published input/output token prices. These are benchmark-based projections, not production load-test results.

For a fresh session, make sure benchmark_results.json is available beside the notebook or in docs/A7-part1/input/.

## 1. Configuration

All settings are embedded below. Keep the common model registry, prompts, decoding policy, token limits, and scoring criteria aligned across teammates. `SELECTED_MODELS` controls which assigned models this invocation runs; an empty list selects all 15 and does not alter the shared registry or input signature.

Protocol version 2 explicitly fixes `do_sample=False`, `num_beams=1`, and `num_return_sequences=1`. Changing `experiment_name` creates a new signature. Increment `protocol_version` when changing generation logic; source-code changes are not detected automatically. Repackaging the existing runner into notebook cells does not change its generation protocol.

Token limits are prepared defaults, not limits verified by real generation. Do not silently change precision, truncate prompts, or retry low-quality answers to improve scores.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
from datetime import datetime, timezone
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import shutil
import statistics
import subprocess
import sys
import time

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE_DIR = Path('/content') if IN_COLAB else Path.cwd()

CONFIG = {'protocol_version': 2,
 'experiment_name': 'a7-part1-merged-en-v1',
 'configuration_status': 'Prepared defaults; hardware fit and output limits not verified by inference.',
 'prompt_file': 'a7_prompts.json',
 'output_root': 'results',
 'cache_dir': '.a7_model_cache',
 'keep_model_cache': False,
 'decoding': 'greedy',
 'seed': 42,
 'repeats': 1,
 'max_input_tokens': 4096,
 'warmup_tokens': 4,
 'max_new_tokens': {'classification': 64,
                    'price_prediction': 128,
                    'text_generation': 256,
                    'structured_information_extraction': 384,
                    'purchase_considerations': 512},
 'models': [{'id': 'Qwen/Qwen3-0.6B',
             'category': 'Ultra-Light (<1B)',
             'params_b': 0.6,
             'quant': 'none',
             'dtype': 'float16',
             'gated': False,
             'template_kwargs': {'enable_thinking': False},
             'revision': None},
            {'id': 'HuggingFaceTB/SmolLM2-360M-Instruct',
             'category': 'Ultra-Light (<1B)',
             'params_b': 0.36,
             'quant': 'none',
             'dtype': 'float16',
             'gated': False,
             'template_kwargs': {},
             'revision': None},
            {'id': 'google/gemma-3-270m-it',
             'category': 'Ultra-Light (<1B)',
             'params_b': 0.27,
             'quant': 'none',
             'dtype': 'float32',
             'gated': True,
             'template_kwargs': {},
             'revision': None},
            {'id': 'meta-llama/Llama-3.2-1B-Instruct',
             'category': 'Small (1-3B)',
             'params_b': 1.24,
             'quant': 'none',
             'dtype': 'float16',
             'gated': True,
             'template_kwargs': {},
             'revision': None},
            {'id': 'Qwen/Qwen3-1.7B',
             'category': 'Small (1-3B)',
             'params_b': 1.7,
             'quant': 'none',
             'dtype': 'float16',
             'gated': False,
             'template_kwargs': {'enable_thinking': False},
             'revision': '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'},
            {'id': 'HuggingFaceTB/SmolLM2-1.7B-Instruct',
             'category': 'Small (1-3B)',
             'params_b': 1.71,
             'quant': 'none',
             'dtype': 'float16',
             'gated': False,
             'template_kwargs': {},
             'revision': None},
            {'id': 'meta-llama/Llama-3.2-3B-Instruct',
             'category': 'Medium (3-5B)',
             'params_b': 3.21,
             'quant': 'none',
             'dtype': 'float16',
             'gated': True,
             'template_kwargs': {},
             'revision': '0cb88a4f764b7a12671c53f0838cd831a0843b95'},
            {'id': 'microsoft/Phi-4-mini-instruct',
             'category': 'Medium (3-5B)',
             'params_b': 3.84,
             'quant': 'none',
             'dtype': 'float16',
             'gated': False,
             'template_kwargs': {},
             'revision': 'cfbefacb99257ffa30c83adab238a50856ac3083'},
            {'id': 'Qwen/Qwen3-4B',
             'category': 'Medium (3-5B)',
             'params_b': 4.02,
             'quant': 'none',
             'dtype': 'float16',
             'gated': False,
             'template_kwargs': {'enable_thinking': False},
             'revision': None},
            {'id': 'Qwen/Qwen2.5-7B-Instruct',
             'category': 'Large (6-9B)',
             'params_b': 7.62,
             'quant': 'nf4',
             'dtype': 'float16',
             'gated': False,
             'template_kwargs': {},
             'revision': None},
            {'id': 'meta-llama/Llama-3.1-8B-Instruct',
             'category': 'Large (6-9B)',
             'params_b': 8.03,
             'quant': 'nf4',
             'dtype': 'float16',
             'gated': True,
             'template_kwargs': {},
             'revision': None},
            {'id': 'mistralai/Mistral-7B-Instruct-v0.3',
             'category': 'Large (6-9B)',
             'params_b': 7.25,
             'quant': 'nf4',
             'dtype': 'float16',
             'gated': False,
             'template_kwargs': {},
             'revision': None},
            {'id': 'Qwen/Qwen2.5-14B-Instruct',
             'category': 'X-Large (10-15B)',
             'params_b': 14.77,
             'quant': 'nf4',
             'dtype': 'float16',
             'gated': False,
             'template_kwargs': {},
             'revision': None},
            {'id': 'microsoft/phi-4',
             'category': 'X-Large (10-15B)',
             'params_b': 14.66,
             'quant': 'nf4',
             'dtype': 'float16',
             'gated': False,
             'template_kwargs': {},
             'revision': None},
            {'id': 'mistralai/Mistral-Nemo-Instruct-2407',
             'category': 'X-Large (10-15B)',
             'params_b': 12.25,
             'quant': 'nf4',
             'dtype': 'float16',
             'gated': False,
             'template_kwargs': {},
             'revision': None}]}

PROMPT_PATH = BASE_DIR / CONFIG['prompt_file']
SELECTED_MODELS = []  # Example: ['Qwen/Qwen3-0.6B']; empty means all 15.
TASK_TYPES = {'classification', 'price_prediction', 'text_generation',
              'structured_information_extraction', 'purchase_considerations'}
CACHE_MARKER = '.a7-owned-cache'
print('Prompt JSON:', PROMPT_PATH)
print('Protocol:', CONFIG['protocol_version'])


## 2. Dependencies: install only when needed

Keep the runtime's CUDA-enabled PyTorch. These packages retain the A6 Transformers 4.x route; use a separate environment from the Week 6 update's Transformers 5.2.0 scripts. Set `INSTALL_DEPENDENCIES=True` when ready. If older ML versions have already been imported, restart the kernel after installation and execute the cells again with installation disabled.

The offline plan does not require these packages. No external requirements file is read.


In [ ]:
INSTALL_DEPENDENCIES = True
REQUIREMENTS = ['transformers==4.57.6',
 'huggingface_hub==0.36.2',
 'accelerate>=1,<2',
 'bitsandbytes>=0.46',
 'sentencepiece>=0.2',
 'protobuf>=4.25',
 'safetensors>=0.4']

if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS], check=True)
    print('Dependencies installed. Restart the kernel if previously imported versions changed.')
else:
    print('Installation skipped. Offline planning needs only the Python standard library.')


## 3. Prompt JSON

The supplied `a7_prompts.json` contains the complete prompts and separate evaluation references. The runner sends only each record's `model_prompt` to the model. `reference` and source metadata stay outside model messages. It does not read a separate `listings.json` or historical price CSV.

In Colab, upload `a7_prompts.json` if it is absent. Locally, place it in the current working directory or edit `PROMPT_PATH` in Section 1. The offline plan later validates the 25 unique prompt IDs and five prompts per task type.


In [ ]:
if not PROMPT_PATH.is_file() and IN_COLAB:
    print('Upload a7_prompts.json.')
    uploaded = files.upload()
    if PROMPT_PATH.name not in uploaded:
        raise FileNotFoundError('The upload must include a7_prompts.json.')
    PROMPT_PATH.parent.mkdir(parents=True, exist_ok=True)
    PROMPT_PATH.write_bytes(uploaded[PROMPT_PATH.name])
    del uploaded
if not PROMPT_PATH.is_file():
    raise FileNotFoundError(f'Place a7_prompts.json at {PROMPT_PATH}, or edit PROMPT_PATH.')
print('Using:', PROMPT_PATH)


## 4. Results and model cache

Drive is recommended for Colab so answers survive session resets. Set `USE_GOOGLE_DRIVE=False` to use temporary Colab storage instead; copy results elsewhere before the session ends. Mounting Drive does not load a model.

The dedicated model cache stays on local disk and is cleared after each model by default. It never targets the shared Hugging Face cache. Do not let two writers use the same run folder concurrently. Teammates running separately should use different result roots and retain each run's manifest.


In [ ]:
USE_GOOGLE_DRIVE = True
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_ROOT = Path('/content/drive/MyDrive/INFO490_A7/results')
else:
    RESULTS_ROOT = BASE_DIR / CONFIG['output_root']
MODEL_CACHE = Path('/content/a7_model_cache') if IN_COLAB else BASE_DIR / CONFIG['cache_dir']
print('Results:', RESULTS_ROOT)
print('Dedicated model cache:', MODEL_CACHE)


## 5. Input handling and experiment identity

Execute the following definition cells in order. They define functions without loading models. This cell validates the bank and registry, builds the experiment signature, and renders a single user message through each model's chat template.

Only `model_prompt` enters the message. References do not affect the generation signature, while prompts, model settings, experiment name, and protocol version do.


In [ ]:
def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)

def validate_bank(bank):
    if not isinstance(bank, list) or len(bank) != 25:
        raise ValueError("Expected one JSON array containing exactly 25 prompts.")
    ids = []
    for record in bank:
        if not isinstance(record, dict):
            raise ValueError("Each prompt record must be an object.")
        for key in ("prompt_id", "task_type", "model_prompt"):
            if not isinstance(record.get(key), str) or not record[key].strip():
                raise ValueError(f"Missing nonempty string: {key}")
        if not isinstance(record.get("reference"), dict) or not record["reference"]:
            raise ValueError(f"Empty reference: {record['prompt_id']}")
        ids.append(record["prompt_id"])
    if len(ids) != len(set(ids)):
        raise ValueError("prompt_id must be unique.")
    counts = Counter(record["task_type"] for record in bank)
    if set(counts) != TASK_TYPES or any(n != 5 for n in counts.values()):
        raise ValueError(f"Expected five prompts per agreed task type; found {dict(counts)}")
    return bank

def load_inputs(config, prompt_path):
    config = dict(config)
    bank = validate_bank(json.loads(Path(prompt_path).read_text(encoding="utf-8-sig")))
    models = config["models"]
    if len(models) != 15 or len({m["id"] for m in models}) != 15:
        raise ValueError("The model registry must contain 15 unique A6 models.")
    if set(Counter(m["category"] for m in models).values()) != {3}:
        raise ValueError("Each model-size category must contain three models.")
    if len({m["category"] for m in models}) != 5:
        raise ValueError("Expected five model-size categories.")
    if config.get("decoding") != "greedy":
        raise ValueError("This runner implements greedy decoding only.")
    for model in models:
        if model.get("quant") not in {"none", "nf4"} or model.get("dtype") not in {"float16", "float32"}:
            raise ValueError(f"Unsupported precision configuration: {model['id']}")
    if set(config["max_new_tokens"]) != TASK_TYPES:
        raise ValueError("Define max_new_tokens for all five actual task types.")
    for value in [config["repeats"], config["max_input_tokens"], config["warmup_tokens"],
                  *config["max_new_tokens"].values()]:
        if type(value) is not int or value < 1:
            raise ValueError("Token limits and repeats must be positive integers.")
    return config, bank

def input_signature(config, bank):
    # References/score changes do not change what the model was asked to generate.
    generation = {key: config.get(key) for key in
                  ("experiment_name", "models", "seed", "repeats", "max_input_tokens", "warmup_tokens",
                   "max_new_tokens", "decoding", "protocol_version")}
    prompts = [{key: record[key] for key in ("prompt_id", "task_type", "model_prompt")}
               for record in bank]
    return digest({"generation": generation, "prompts": prompts})


def render_prompt(tokenizer, record, spec):
    text = record["model_prompt"]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": text}], tokenize=False,
            add_generation_prompt=True, **spec.get("template_kwargs", {}))
    return text

## 6. Per-answer persistence and resume

Each generated answer is written and flushed before scoring. A refusal, invalid JSON response, or low-quality answer still counts as a completed generation and is not automatically rerun. Errors are logged separately. Resume can preserve and repair an interrupted final JSONL fragment; corrupt middle records cause an error.

The cache helper verifies its deletion target is a direct child of the dedicated, marked cache directory.


In [ ]:
def read_jsonl(path, repair_tail=False):
    path = Path(path)
    if not path.exists():
        return []
    content = path.read_bytes()
    lines = content.splitlines(keepends=True)
    records, offset = [], 0
    for index, line in enumerate(lines):
        if line.strip():
            try:
                value = json.loads(line)
                if not isinstance(value, dict):
                    raise ValueError("JSONL records must be objects")
                records.append(value)
            except (ValueError, UnicodeDecodeError) as exc:
                # Only an interrupted, unterminated final line can be repaired.
                if repair_tail and index == len(lines) - 1 and not line.endswith(b"\n"):
                    backup = path.with_name(path.name + f".partial-{time.time_ns()}")
                    backup.write_bytes(line)
                    with path.open("r+b") as handle:
                        handle.truncate(offset)
                        handle.flush()
                        os.fsync(handle.fileno())
                    print(f"Preserved interrupted tail: {backup.name}")
                    break
                raise ValueError(f"Invalid JSONL at {path}, line {index + 1}") from exc
        offset += len(line)
    return records


def append_jsonl(path, record):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    separator = b""
    if path.exists() and path.stat().st_size:
        with path.open("rb") as handle:
            handle.seek(-1, os.SEEK_END)
            if handle.read(1) != b"\n":
                separator = b"\n"
    payload = json.dumps(record, ensure_ascii=False).encode("utf-8") + b"\n"
    with path.open("ab") as handle:
        handle.write(separator + payload)
        handle.flush()
        os.fsync(handle.fileno())


def completed_keys(rows):
    return {(r["model_id"], r["prompt_id"], r["repeat"]) for r in rows
            if r.get("status") == "ok"}


def purge_model_cache(root, model_id):
    root = Path(root).resolve()
    if not (root / CACHE_MARKER).is_file():
        raise ValueError("Refusing to purge a cache not marked as A7-owned.")
    name = "models--" + model_id.replace("/", "--")
    target = root / name
    # Verify the absolute deletion target is exactly this repository's child.
    if target.resolve().parent != root or target.name != name or target.is_symlink():
        raise ValueError("Cache deletion target escaped the dedicated cache.")
    if target.is_dir():
        shutil.rmtree(target)


def execute_pending(tasks, generate, raw_path, error_path, common, stop_on=()):
    for record, repeat in tasks:
        identity = {"prompt_id": record["prompt_id"], "task_type": record["task_type"],
                    "repeat": repeat}
        try:
            answer = generate(record)
            elapsed = answer["latency_s"]
            row = {**common, **identity, **answer, "status": "ok",
                   "model_prompt": record["model_prompt"],
                   "tokens_per_sec": answer["output_tokens"] / elapsed if elapsed > 0 else None,
                   "timestamp_utc": datetime.now(timezone.utc).isoformat()}
            # No parser or grader can discard an answer before it is persisted.
            append_jsonl(raw_path, row)
            print(f"{common['model_id']} {record['prompt_id']} {elapsed:.3f}s "
                  f"output_tokens={answer['output_tokens']}")
        except Exception as exc:
            append_jsonl(error_path, {**common, **identity, "status": "error",
                                     "stage": "generation", "error_type": type(exc).__name__,
                                     "error": str(exc),
                                     "timestamp_utc": datetime.now(timezone.utc).isoformat()})
            print(f"FAILED {common['model_id']} {record['prompt_id']}: {type(exc).__name__}")
            if isinstance(exc, stop_on):
                break


## 7. Export and manual-score preservation

Initial exports create `results.csv`, a blank `scores.csv` template, and `summary.csv`. Re-export preserves recorded scores and adds missing answer rows; it does not score model responses or apply review corrections automatically. For the completed run, an AI-assisted scoring pass followed by targeted user confirmations populated all 375 scores. Summary rows include missing-answer and scored-answer counts; partially scored results would not support a complete model ranking.


In [ ]:
def _write_csv(path, fields, rows):
    with Path(path).open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def export_results(run_dir, bank, config):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    rows = read_jsonl(run_dir / "raw_outputs.jsonl")
    unique = {}
    for row in rows:
        if row.get("status") == "ok":
            unique[(row["model_id"], row["prompt_id"], row["repeat"])] = row
    rows = list(unique.values())
    model_map = {m["id"]: m for m in config["models"]}
    fields = ["run_id", "model_id", "model_revision", "model_category", "params_b", "quant",
              "dtype", "prompt_id", "task_type", "repeat", "prompt_tokens", "output_tokens",
              "latency_s", "tokens_per_sec", "peak_vram_gb", "max_new_tokens",
              "hit_token_limit", "stop_reason", "status", "raw_output"]
    _write_csv(run_dir / "results.csv", fields, rows)

    score_path = run_dir / "scores.csv"
    score_fields = ["run_id", "model_id", "model_revision", "prompt_id", "task_type", "repeat",
                    "scorer", "self_evaluation_score", "score_details", "score_notes"]
    existing = []
    if score_path.exists():
        with score_path.open(encoding="utf-8-sig", newline="") as handle:
            existing = list(csv.DictReader(handle))
    existing_keys = {(r["model_id"], r["prompt_id"], int(r["repeat"])) for r in existing}
    additions = [{key: row.get(key, "") for key in score_fields} for row in rows
                 if (row["model_id"], row["prompt_id"], row["repeat"]) not in existing_keys]
    # Never overwrite existing human entries. Later generations add blank rows only.
    if additions:
        with score_path.open("a", encoding="utf-8-sig", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=score_fields, extrasaction="ignore")
            if not score_path.exists() or score_path.stat().st_size == 0:
                writer.writeheader()
            writer.writerows(additions)
    elif not score_path.exists():
        _write_csv(score_path, score_fields, [])

    with score_path.open(encoding="utf-8-sig", newline="") as handle:
        score_rows = list(csv.DictReader(handle))
    scores = {}
    for row in score_rows:
        value = row.get("self_evaluation_score", "").strip()
        if value:
            number = float(value)
            if not math.isfinite(number) or not 1 <= number <= 10:
                raise ValueError(f"Score must be between 1 and 10: {row['prompt_id']}")
            scores[(row["model_id"], row["prompt_id"], int(row["repeat"]))] = number

    summary = []
    groups = {kind: [r for r in bank if r["task_type"] == kind] for kind in sorted(TASK_TYPES)}
    groups["ALL"] = bank
    for model_id, spec in model_map.items():
        for kind, prompts in groups.items():
            ids = {p["prompt_id"] for p in prompts}
            selected = [r for r in rows if r["model_id"] == model_id and r["prompt_id"] in ids]
            expected = len(prompts) * config["repeats"]
            graded = [scores[k] for k in unique if k[0] == model_id and k[1] in ids and k in scores]
            latencies = [r["latency_s"] for r in selected]
            speeds = [r["tokens_per_sec"] for r in selected if r.get("tokens_per_sec") is not None]
            mean = lambda values: statistics.mean(values) if values else ""
            summary.append({"model_id": model_id, "model_category": spec.get("category", ""),
                            "task_type": kind, "expected_responses": expected,
                            "completed_responses": len(selected), "missing_responses": expected-len(selected),
                            "scored_responses": len(graded), "self_evaluation_mean": mean(graded),
                            "latency_median_s": statistics.median(latencies) if latencies else "",
                            "tokens_per_sec_mean": mean(speeds),
                            "prompt_tokens_mean": mean([r["prompt_tokens"] for r in selected]),
                            "output_tokens_mean": mean([r["output_tokens"] for r in selected]),
                            "token_limit_hits": sum(bool(r.get("hit_token_limit")) for r in selected)})
    _write_csv(run_dir / "summary.csv", list(summary[0]), summary)
    return {"completed": len(rows), "expected": len(model_map)*len(bank)*config["repeats"],
            "scored": len(scores)}


## 8. CUDA generation backend

Defining this class does not import ML packages. Only an explicit generation call instantiates it. Models load sequentially; the larger registry entries retain NF4 quantization. Inputs exceeding the allowed context raise an error instead of being silently truncated.

Generation uses the fixed greedy parameters. CUDA synchronization brackets the generation call; reported latency excludes model download, loading, tokenization, and warm-up. Output-token counts exclude input tokens. Hitting the output limit is recorded for later inspection and does not trigger an automatic retry.


In [ ]:
class CudaBackend:
    """The only inference backend; instantiated only when RUN_GENERATION is enabled."""

    def __init__(self, config, cache_dir):
        import torch
        import transformers
        from huggingface_hub import HfApi, snapshot_download
        self.torch, self.transformers = torch, transformers
        self.snapshot_download = snapshot_download
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA GPU required. In Colab choose a GPU runtime; no CPU fallback.")
        self.token = os.environ.get("HF_TOKEN") or None
        self.api = HfApi(token=self.token)
        self.config, self.cache_dir = config, Path(cache_dir).resolve()
        self.model, self.tokenizer = None, None
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        marker = self.cache_dir / CACHE_MARKER
        if not marker.exists() and any(self.cache_dir.iterdir()):
            raise ValueError("Use an empty dedicated A7 cache, not a shared HF cache.")
        marker.write_text("A7 dedicated model cache", encoding="utf-8")
        versions = {}
        for package in ("torch", "transformers", "huggingface-hub", "accelerate", "bitsandbytes"):
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = "not installed"
        gpu = torch.cuda.get_device_properties(0)
        self.runtime = {"packages": versions, "gpu_name": gpu.name,
                        "gpu_memory_bytes": gpu.total_memory, "cuda_version": torch.version.cuda,
                        "attention": "sdpa", "device": "cuda:0"}

    def load(self, spec, revision):
        required_gb = spec["params_b"] * 2 * 1.25
        free_gb = shutil.disk_usage(self.cache_dir).free / 1e9
        repo_cache = self.cache_dir / ("models--" + spec["id"].replace("/", "--"))
        if not repo_cache.exists() and free_gb < required_gb:
            raise RuntimeError(f"Approximate checkpoint disk budget {required_gb:.1f} GB; "
                               f"only {free_gb:.1f} GB free (not a VRAM estimate).")
        local = self.snapshot_download(
            repo_id=spec["id"], revision=revision, token=self.token, cache_dir=str(self.cache_dir),
            allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "*.jinja", "*.tiktoken"],
            ignore_patterns=["original/*", "onnx/*"])
        self.tokenizer = self.transformers.AutoTokenizer.from_pretrained(local, local_files_only=True)
        dtype = self.torch.float32 if spec["dtype"] == "float32" else self.torch.float16
        kwargs = {"local_files_only": True, "torch_dtype": dtype,
                  "attn_implementation": "sdpa", "low_cpu_mem_usage": True}
        if spec["quant"] == "nf4":
            kwargs["quantization_config"] = self.transformers.BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=self.torch.float16, bnb_4bit_use_double_quant=True)
            kwargs["device_map"] = {"": 0}
        self.model = self.transformers.AutoModelForCausalLM.from_pretrained(local, **kwargs)
        if spec["quant"] != "nf4":
            self.model = self.model.to("cuda:0")
        self.model.eval()

    def generate(self, spec, record, max_tokens):
        prompt = render_prompt(self.tokenizer, record, spec)
        inputs = self.tokenizer(prompt, return_tensors="pt",
                                add_special_tokens=not bool(self.tokenizer.chat_template)).to("cuda:0")
        n_input = int(inputs["input_ids"].shape[1])
        context = getattr(self.model.config, "max_position_embeddings", None)
        if n_input > self.config["max_input_tokens"] or (context and n_input + max_tokens > context):
            raise ValueError(f"Input length {n_input}, output allowance {max_tokens}: "
                             "context budget exceeded; no truncation applied.")
        self.transformers.set_seed(self.config["seed"])
        self.torch.cuda.reset_peak_memory_stats()
        self.torch.cuda.synchronize()
        start = time.perf_counter()
        with self.torch.inference_mode():
            output = self.model.generate(
                **inputs, max_new_tokens=max_tokens, do_sample=False,
                num_beams=1, num_return_sequences=1,
                temperature=None, top_p=None, top_k=None,
                pad_token_id=self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None
                else self.tokenizer.eos_token_id)
        self.torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        new = output[0, n_input:]
        count = int(new.numel())
        eos = self.model.generation_config.eos_token_id
        eos = eos if isinstance(eos, list) else [eos]
        stop = "eos" if count and int(new[-1].item()) in eos else ("length" if count >= max_tokens else "unknown")
        return {"raw_output": self.tokenizer.decode(new, skip_special_tokens=True),
                "prompt_tokens": n_input, "output_tokens": count, "latency_s": elapsed,
                "peak_vram_gb": self.torch.cuda.max_memory_allocated()/1e9,
                "max_new_tokens": max_tokens, "hit_token_limit": count >= max_tokens,
                "stop_reason": stop, "rendered_prompt_sha256": digest(prompt)}

    def close_model(self):
        self.model, self.tokenizer = None, None
        gc.collect()
        self.torch.cuda.empty_cache()


## 9. Sequential model runner

This function resolves model revisions, records the actual runtime, and selects a run directory from the input and runtime signatures. It loads one model, performs an unscored warm-up, generates pending answers, exports results, and releases the model before moving on.

Resume skips saved answers with the same run identity. GPU or software changes create a separate run. An out-of-memory error stops the remaining questions for that model; the failure is recorded and the runner continues to the next model without silently changing precision or limits.


In [ ]:
def run_experiment(config, bank, output_root, cache_dir, selected_ids=None):
    backend = CudaBackend(config, cache_dir)
    signature = input_signature(config, bank)
    run_id = digest({"input_signature": signature, "runtime": backend.runtime})[:16]
    run_dir = Path(output_root).resolve() / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = run_dir / "run_config.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    else:
        manifest = {"run_id": run_id, "input_signature": signature,
                    "runtime": backend.runtime, "config": config, "resolved_revisions": {}}
        write_json(manifest_path, manifest)
    # Evaluation references are kept on disk only, versioned independently of inference.
    write_json(run_dir / f"prompt_bank_{digest(bank)[:12]}.json", bank)
    raw_path, error_path = run_dir / "raw_outputs.jsonl", run_dir / "errors.jsonl"
    done = completed_keys(read_jsonl(raw_path, repair_tail=True))
    read_jsonl(error_path, repair_tail=True)
    print(f"Run directory: {run_dir}\nActual GPU: {backend.runtime['gpu_name']}\n"
          f"Completed before this invocation: {len(done)}")
    for spec in config["models"]:
        if selected_ids and spec["id"] not in selected_ids:
            continue
        tasks = [(record, repeat) for repeat in range(config["repeats"]) for record in bank
                 if (spec["id"], record["prompt_id"], repeat) not in done]
        if not tasks:
            print("Already complete:", spec["id"])
            continue
        common = {"run_id": run_id, "model_id": spec["id"], "model_category": spec["category"],
                  "params_b": spec["params_b"], "quant": spec["quant"], "dtype": spec["dtype"]}
        try:
            revision = manifest["resolved_revisions"].get(spec["id"])
            if not revision:
                revision = backend.api.model_info(spec["id"], revision=spec.get("revision") or "main").sha
                manifest["resolved_revisions"][spec["id"]] = revision
                write_json(manifest_path, manifest)
            common["model_revision"] = revision
            backend.load(spec, revision)
            backend.generate(spec, tasks[0][0], config["warmup_tokens"])
            generate = lambda record: backend.generate(spec, record, config["max_new_tokens"][record["task_type"]])
            execute_pending(tasks, generate, raw_path, error_path, common,
                            stop_on=(backend.torch.cuda.OutOfMemoryError,))
        except Exception as exc:
            append_jsonl(error_path, {**common, "status": "error", "stage": "load_or_warmup",
                                     "error_type": type(exc).__name__, "error": str(exc),
                                     "timestamp_utc": datetime.now(timezone.utc).isoformat()})
            print(f"MODEL FAILED {spec['id']}: {type(exc).__name__}: {exc}")
        finally:
            backend.close_model()
            if not config.get("keep_model_cache", False):
                purge_model_cache(cache_dir, spec["id"])
            status = export_results(run_dir, bank, config)
            print("Recorded:", status)
    status = export_results(run_dir, bank, config)
    print(f"Generation coverage: {status['completed']}/{status['expected']}. "
          "Recorded answers are not correctness or quality passes.")
    return run_dir


## 10. Offline plan

This reads the prompt JSON and validates the embedded configuration. It does not contact Hugging Face, import ML packages, initialize CUDA, or perform inference. The full default should show five tasks with five prompts each, 15 models, and 375 planned answers.


In [ ]:
config, bank = load_inputs(CONFIG, PROMPT_PATH)
if SELECTED_MODELS and not set(SELECTED_MODELS) <= {m['id'] for m in config['models']}:
    raise ValueError('SELECTED_MODELS includes an unknown model ID.')
selected = SELECTED_MODELS or [m['id'] for m in config['models']]
print(json.dumps({
    'mode': 'offline_plan_no_downloads_no_inference',
    'prompts': len(bank),
    'task_counts': dict(Counter(record['task_type'] for record in bank)),
    'selected_models': selected,
    'repeats': config['repeats'],
    'planned_responses': len(selected) * len(bank) * config['repeats'],
    'max_new_tokens': config['max_new_tokens'],
    'input_signature': input_signature(config, bank)
}, indent=2))


## 11. Explicit generation switch

Before enabling this cell, choose a CUDA GPU runtime, install the dependencies, and obtain access to gated model repositories. An HF token is read from Colab secrets or the environment, with a hidden input fallback. Do not paste a token into notebook code.

Set `RUN_GENERATION=True` only when ready to download model weights and generate answers. Each selected model receives all 25 prompts. To resume, retain the prior results directory, use the same prompt/configuration and runtime conditions, and run the notebook again. A fresh session may redownload a model, but completed answers in the matching run directory are preserved.

The default remains disabled. This notebook does not claim that every model fits the available GPU or that token limits prevent all truncation.


In [ ]:
RUN_GENERATION = True
RUN_DIR = None

if RUN_GENERATION:
    from getpass import getpass
    config, bank = load_inputs(CONFIG, PROMPT_PATH)
    if SELECTED_MODELS and not set(SELECTED_MODELS) <= {m['id'] for m in config['models']}:
        raise ValueError('SELECTED_MODELS includes an unknown model ID.')
    token = os.environ.get('HF_TOKEN')
    if IN_COLAB and not token:
        from google.colab import userdata
        try:
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass('HF token (hidden; blank to proceed without): ').strip()
    if token:
        os.environ['HF_TOKEN'] = token
    RUN_DIR = run_experiment(config, bank, RESULTS_ROOT, MODEL_CACHE,
                             set(SELECTED_MODELS) if SELECTED_MODELS else None)
    print('Run directory:', RUN_DIR)
else:
    print('Generation disabled. No model was downloaded or run by this cell.')


## 12. AI-assisted scoring, targeted review, and re-export

The first scoring pass was **AI-assisted**: At the **global level**, I used AI to evaluate all 375 saved responses from 15 models using the prepared prompt references and task-specific scoring criteria. AI helped apply consistent evaluation standards, calculate weighted component scores, and organize results across models and task types. For example, price prediction responses were evaluated based on price accuracy, factual grounding, reasoning, and format compliance, while purchase considerations used a weighted 4+3+2+1 scoring system. AI also helped identify disputed scores and potential inconsistencies for further review.

At the **local level**, I used AI to examine selected responses more closely by comparing the original prompts, references, model outputs, and scoring details. For example, the Phi-4 response to CLS01 was corrected from 8 to 9 after recognizing that its identifiable answer deserved partial format credit. Several price prediction scores were also corrected because the abbreviation "U.S." had been incorrectly treated as a sentence boundary. For RISK05, I used AI to reassess the affected responses under a consistent interpretation of the high-temperature requirement. I reviewed the proposed adjustments and confirmed the final scores without regenerating model responses.

The final 1–10 scores, component details, and supporting reasons are recorded in `scores.csv`. The corrected scores are used to update `summary.csv`, while the original responses, generation settings, and performance measurements remain unchanged. Re-exporting the results does not require loading or rerunning any models.
Import the corrected scoring files into the exact saved run directory **before** setting `EXPORT_RESULTS=True`. Re-export preserves the scores already in that directory and refreshes summaries without loading a model; it does not independently merge confirmation logs or apply corrections. In a fresh Colab session, set `SCORED_RUN_DIR` to the prior run path. Set `EXPORT_RESULTS=False` again after the update.

| Output | Purpose |
|---|---|
| `run_config.json` | Protocol, settings, actual runtime, and resolved model revisions. |
| `prompt_bank_<hash>.json` | Evaluation-side prompt-bank snapshot with references. |
| `raw_outputs.jsonl` | Original answers, token counts, timing, and generation metadata. |
| `errors.jsonl` | Recorded loading, warm-up, or generation errors when present. |
| `results.csv` | Readable per-answer results. |
| `scores.csv` | Final grades, component details, review provenance, and supporting notes. |
| `summary.csv` | Coverage, original performance records, and corrected quality means. |

These are recorded outputs, not extra inputs to the generation runner.

In [ ]:
EXPORT_RESULTS = True
SCORED_RUN_DIR = False  # Example: RESULTS_ROOT / 'the_actual_run_id'

if EXPORT_RESULTS:
    target = SCORED_RUN_DIR or RUN_DIR
    if target is None:
        raise ValueError('Set SCORED_RUN_DIR to the exact run directory first.')
    target = Path(target)
    config, bank = load_inputs(CONFIG, PROMPT_PATH)
    manifest = json.loads((target / 'run_config.json').read_text(encoding='utf-8'))
    if manifest['input_signature'] != input_signature(config, bank):
        raise ValueError("The supplied bank/config does not match this run's generation inputs.")
    print(export_results(target, bank, manifest['config']))
else:
    print('No re-export requested. Fill actual scores after answers are available.')


## 13. Evaluation framework and scoring-stage clarifications

The task-level dimensions and weights below describe the evaluation framework. Some interpretation details were clarified during AI-assisted scoring and targeted review, **after generation**, as documented in Section 12. They must not be represented as changes to the pre-generation `model_prompt` or `reference`. Score only saved responses on a **1–10 scale**; leave missing or failed generations unscored. The references remain evaluation-only inputs.

| Task | Scoring definition |
|---|---|
| **Classification (CLS01–05)** | **Category: 8 or 0** for one category exactly matching the reference, including case. **Format: 2** for the required JSON object, **1** for an identifiable answer with minor format issues, **0** for unusable structure. Final score = max(1, sum). Judge category and format separately. |
| **Price prediction (PP01–05)** | Average four **1–10** scores: price accuracy, information grounding and honesty, grounded reasoning, and format compliance. Accept justified uncertainty without invented attributes. Use the price rule below. |
| **Text generation (TG01–05)** | Average four **1–10** scores: information accuracy, appeal and clarity, factual grounding, and format compliance. Use only supplied facts; require a listing description under 100 words. Short, accurate descriptions can receive full marks. |
| **Structured extraction (SIE01–05)** | Average six **1–10** scores: constraint accuracy, taxonomy mapping/query preservation, ambiguity handling, fulfillment extraction, field-value validity, and JSON compliance. Accept equivalent wording and array order; do not score fulfillment priority. |
| **Purchase considerations (RISK01–05)** | **Grounding 0–4 + relevance/usefulness 0–3 + known/unknown distinction 0–2 + format 0–1.** Final score = max(1, sum). Apply the anchors below. |

**Price accuracy.** The provisional USD intervals are PP01 **58–94**, PP02 **54–90**, PP03 **10–35**, PP04 **8–33**, and PP05 **6–16**. Award **10** inside the inclusive interval. Outside it, calculate `d = 100 × distance to the nearest boundary / interval midpoint` and use the first matching upper bound:

| Upper bound for d (%) | 5 | 10 | 15 | 20 | 27.5 | 35 | 42.5 | 50 | >50 |
|---|---|---|---|---|---|---|---|---|---|
| Score | 9 | 8 | 7 | 6 | 5 | 4 | 3 | 2 | 1 |

These historical-data ranges are approximate references, not definitive current market values. PP04 has weaker comparisons, including different conditions and a mug set. Comparable-item details remain in `reference`; no additional source file is required at runtime.

**Purchase-consideration anchors:**

- **Grounding (4→0):** fully supported; minor inaccuracy; consequential error; extensive unsupported claims; little reliable grounding.
- **Relevance (3→0):** both required points covered with useful, preference-aware advice; one point missing or weak prioritization; mostly generic/repetitive content; off-task.
- **Known/unknown distinction (2→0):** clear separation; local confusion; repeated unsupported certainty.
- **Format (1/0):** exactly the requested JSON keys with short English string arrays, or noncompliance. Cap the total at **4** if an unsupported serious-defect claim or safety guarantee materially distorts the purchase judgment.

**Shared interpretation.** Evaluate meaning and evidence, not keyword counts. Required points are semantic requirements; acceptable points are optional, and concise answers can earn full marks. Do not require unknown flaws, reward unnecessary questions, or treat condition labels as functional guarantees. Budget preference changes priorities, not the need to address core concerns. Preserve the extraction prompts' cent-precision boundaries: “under 30USD ” means **29.99**, and “under 100 USD” means **99.99**.

Keep original outputs unchanged. Record the total in `self_evaluation_score`, component scores in `score_details`, and reasons in `score_notes`. Partial-score averages describe only the scored subset, not a complete model ranking.

## 14. Step 1.5: Benchmark Experiment Results

The table displays all 375 saved evaluations and can be exported to CSV. It is loaded entirely from `benchmark_results.json`; no model runs are required.


In [ ]:
# Step 1.5: Complete per-answer benchmark table (no inference)
import csv
import json
from html import escape
from pathlib import Path
from IPython.display import HTML, display

_base = Path(globals().get("BASE_DIR", Path.cwd()))
BENCHMARK_JSON_PATH = _base / "benchmark_results.json"
if not BENCHMARK_JSON_PATH.is_file():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(f"Place benchmark_results.json at {BENCHMARK_JSON_PATH}") from exc
    print("Upload benchmark_results.json (no GPU needed).")
    _uploaded = files.upload()
    if "benchmark_results.json" not in _uploaded:
        raise FileNotFoundError("benchmark_results.json was not uploaded.")
    BENCHMARK_JSON_PATH.write_bytes(_uploaded["benchmark_results.json"])

with BENCHMARK_JSON_PATH.open(encoding="utf-8") as _f:
    BENCHMARK_DATA = json.load(_f)
BENCHMARK_RECORDS = BENCHMARK_DATA["records"]

RESULT_COLUMNS = [
    "Model", "Model Category", "Tokens/sec", "Latency (s)",
    "Prompt Type", "Prompt ID", "Prompt", "Self-Evaluation Score (1-10)",
]
RESULT_ROWS = [{
    "Model": r["model_id"],
    "Model Category": r["model_category"],
    "Tokens/sec": r["tokens_per_sec_text"],
    "Latency (s)": r["latency_s_text"],
    "Prompt Type": r["task_type"],
    "Prompt ID": r["prompt_id"],
    "Prompt": r["prompt"],
    "Self-Evaluation Score (1-10)": r["score_text"],
} for r in BENCHMARK_RECORDS]

RESULT_CSV_PATH = _base / "A7_Step1_5_Benchmark_375.csv"
with RESULT_CSV_PATH.open("w", encoding="utf-8-sig", newline="") as _f:
    _writer = csv.DictWriter(_f, fieldnames=RESULT_COLUMNS)
    _writer.writeheader()
    _writer.writerows(RESULT_ROWS)

_header = "".join(f"<th>{escape(k)}</th>" for k in RESULT_COLUMNS)
_body = "".join(
    "<tr>" + "".join(f"<td>{escape(str(row[k]))}</td>" for k in RESULT_COLUMNS) + "</tr>"
    for row in RESULT_ROWS
)
_html = (
    '<div id="a7-step15-results" style="max-height:650px;overflow:auto;border:1px solid #ddd">'
    '<table style="border-collapse:collapse;font-size:12px;min-width:1300px">'
    '<thead style="position:sticky;top:0;background:#eee;color:#222"><tr>'
    + _header + '</tr></thead><tbody>' + _body + '</tbody></table></div>'
    '<style>#a7-step15-results td,#a7-step15-results th{border:1px solid #ddd;padding:6px;text-align:left;vertical-align:top}'
    '#a7-step15-results td:nth-child(7){min-width:420px;max-width:620px;white-space:pre-wrap;word-break:break-word}'
    '#a7-step15-results td:nth-child(1){min-width:240px}</style>'
)
display(HTML(_html))
print(f"Complete records: {len(RESULT_ROWS)}; CSV export: {RESULT_CSV_PATH}")


## 15. Step 1.6: Analysis

Quality is the arithmetic mean of per-response scores; speed is the arithmetic mean of output tokens/sec; latency is the median generation time in seconds. Category results aggregate all 75 responses per category.


In [ ]:
# Step 1.6: Model/category summaries and rubric-grounded analysis
import json
from collections import defaultdict
from html import escape
from pathlib import Path
from statistics import mean, median
from IPython.display import HTML, display

if "BENCHMARK_RECORDS" not in globals():
    _base = Path(globals().get("BASE_DIR", Path.cwd()))
    with (_base / "benchmark_results.json").open(encoding="utf-8") as _f:
        BENCHMARK_RECORDS = json.load(_f)["records"]

_by_model = defaultdict(list)
_by_category = defaultdict(list)
for _r in BENCHMARK_RECORDS:
    _by_model[_r["model_id"]].append(_r)
    _by_category[_r["model_category"]].append(_r)

def _aggregate(records):
    return {
        "mean_quality": mean(float(r["self_evaluation_score"]) for r in records),
        "mean_tokens_per_sec": mean(float(r["tokens_per_sec"]) for r in records),
        "median_latency_s": median(float(r["latency_s"]) for r in records),
        "responses": len(records),
    }

MODEL_SUMMARY = [dict(model_id=name,
                      model_category=records[0]["model_category"],
                      params_b=records[0]["params_b"],
                      **_aggregate(records))
                 for name, records in _by_model.items()]
MODEL_SUMMARY.sort(key=lambda r: r["mean_quality"], reverse=True)
CATEGORY_SUMMARY = [dict(model_category=name, **_aggregate(records))
                    for name, records in _by_category.items()]
CATEGORY_SUMMARY.sort(key=lambda r: r["mean_quality"], reverse=True)

def _table(headers, rows):
    heading = "".join(f"<th>{escape(h)}</th>" for h in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(v))}</td>" for v in row)
                   + "</tr>" for row in rows)
    return ("<table class='a7-summary-table' style='border-collapse:collapse;font-size:13px'>"
            "<thead style='background:#eee;color:#222'><tr>" + heading
            + "</tr></thead><tbody>" + body + "</tbody></table>"
            "<style>.a7-summary-table td,.a7-summary-table th{border:1px solid #ddd;padding:5px;text-align:left}</style>")

display(HTML("<h4>15-Model Summary</h4>" + _table(
    ["Model", "Category", "Parameters (B)", "Mean Quality (1-10)",
     "Mean Tokens/sec", "Median Latency (s)"],
    [[r["model_id"], r["model_category"], f'{r["params_b"]:.2f}',
      f'{r["mean_quality"]:.3f}', f'{r["mean_tokens_per_sec"]:.2f}',
      f'{r["median_latency_s"]:.3f}'] for r in MODEL_SUMMARY])))
display(HTML("<h4>Five Model Categories</h4>" + _table(
    ["Model Category", "Responses", "Mean Quality (1-10)",
     "Mean Tokens/sec", "Median Latency (s)"],
    [[r["model_category"], r["responses"], f'{r["mean_quality"]:.3f}',
      f'{r["mean_tokens_per_sec"]:.2f}', f'{r["median_latency_s"]:.3f}']
     for r in CATEGORY_SUMMARY])))

QUALITY_FLOOR = 8.0
_eligible = [r for r in MODEL_SUMMARY if r["mean_quality"] >= QUALITY_FLOOR]
if not _eligible:
    raise ValueError("No model reaches the quality floor; revise the selection rule explicitly.")
_balance = max(_eligible, key=lambda r: (r["mean_tokens_per_sec"], r["mean_quality"]))
_qwen4 = next(r for r in MODEL_SUMMARY if r["model_id"] == "Qwen/Qwen3-4B")
_phi4 = next(r for r in MODEL_SUMMARY if r["model_id"] == "microsoft/phi-4")
_top = CATEGORY_SUMMARY[0]

**Step 1.6: Analysis**

**1. Which model category produced the best outputs?**

The X-Large (10-15B) category had the highest mean self-evaluation score (8.491/10) across its 75 responses.

**2. Did larger models always perform better?**

No. For example, Qwen3-4B (4.02B parameters, 8.287/10) outscored phi-4 (14.66B parameters, 7.848/10). Category averages show a general size trend, but individual model architecture and task behavior also matter.

**3. Which model balanced speed and output quality best?**

Using a quality floor of 8.0/10 and then maximizing mean output tokens/sec, Qwen/Qwen3-4B was the best fit under this criterion: 8.287/10, 16.51 tokens/sec, and median latency 3.944 s. This is a stated trade-off rule, not a universal ranking.

**Interpretation limits**: Each model answered only 25 prompts in one run. Output length, task mix, model architecture, NF4 quantization for larger models, the shared T4 environment, and AI-assisted scoring can all affect comparisons.

# Part 2: Cost and Scaling Simulation

This section uses the completed Part 1 benchmark to estimate token use, self-hosted capacity, and API cost at the six DAU levels required by the assignment. The 15 models are alternatives; each row assumes one model serves all requests.

**Assumptions**

- Each DAU sends 5 requests per day, split evenly across the five task types.
- Token use is each model's mean across its 25 benchmark answers. AWS capacity uses mean generation latency; median latency is shown as a typical-request reference.
- Part 1 latency times `model.generate()` (prompt prefill and decoding). It excludes prompt rendering, tokenization, model loading, warm-up, network, and queue time, so it is not full request latency.
- AWS requests are spread evenly over 24 hours, with 70% target utilization. Replica count is `ceil(DAU × 5 × mean latency ÷ (86,400 × 0.70))`, with at least one replica. This simple estimate treats a replica as handling one request at a time; it uses the Part 1 T4 benchmark and is not a production load test.
- AWS cost uses one `g4dn.4xlarge` T4 at $1.204/hour. Real throughput may differ.
- **Model matching is model-level and family-first.** I looked for the same model or family first, then a paid model with a similar size and general instruction use. This gives 7 distinct API models for the 15 local models instead of reusing only 4 category-level alternatives. Some local models share one API alternative when they belong to the same family or have similar roles; each row lists the reason. Larger models and different families are labeled, and their costs are approximate proxies.
- HF API costs use public input and output prices per million tokens. The calculation reuses local benchmark token counts; another model or tokenizer may use different counts. HF manages API infrastructure, and actual machine counts and throughput are not public. This compares usage cost, not equal per-replica performance.
- Prices and provider availability were checked against the HF Inference Providers listing on 2026-10-26. Estimates exclude credits, taxes, and discounts.

**Current matching and price references**

| HF API model | Provider | Input ($/1M) | Output ($/1M) | Why it is used |
|---|---|---:|---:|---|
| `Qwen/Qwen3-4B-Instruct-2507` | Nscale | $0.01 | $0.03 | Qwen3 family; close scale to 1.7B–4B local models |
| `ibm-granite/granite-4.2-3b` | DeepInfra | $0.03 | $0.12 | Closest verified small general instruction model for SmolLM; larger and different family |
| `google/gemma-3-4b-it` | DeepInfra | $0.05 | $0.10 | Same Gemma 3 family as the local 270M model; larger alternative |
| `meta-llama/Llama-3.1-8B-Instruct` | Novita | $0.02 | $0.05 | Same Llama family; exact match for one local model and larger alternatives for 1B/3B |
| `Qwen/Qwen3.5-9B` | DeepInfra | $0.10 | $0.15 | Related Qwen family and similar scale to the 7B Qwen/Mistral local models |
| `Qwen/Qwen3-14B` | Nscale | $0.07 | $0.20 | Related Qwen family and similar size to the 14B local model; also a similar-scale alternative for Mistral Nemo |
| `microsoft/phi-4` | DeepInfra | $0.07 | $0.14 | Exact model match for local Phi-4; a similar-scale alternative for Mistral Nemo |

Model availability, provider names, and per-token rates are listed on the [HF Inference Providers model and price list](https://huggingface.co/inference/models). Model pages: [Qwen3 4B Instruct](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507), [Granite 4.2 3B](https://huggingface.co/ibm-granite/granite-4.2-3b), [Gemma 3 4B](https://huggingface.co/google/gemma-3-4b-it), [Llama 3.1 8B](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct), [Qwen3.5 9B](https://huggingface.co/Qwen/Qwen3.5-9B), [Qwen3 14B](https://huggingface.co/Qwen/Qwen3-14B), and [Phi-4](https://huggingface.co/microsoft/phi-4). HF's [pricing and billing page](https://huggingface.co/docs/inference-providers/main/pricing) explains its pay-as-you-go billing.

The mappings are not identical-model comparisons in every case. No currently listed priced option close to SmolLM2's 0.36B–1.7B size was verified, so Granite 3B is a **Larger-model alternative**. The same label applies to Qwen3 4B, Gemma 4B, and Llama 8B alternatives for smaller local models. Qwen3.5 9B is a newer related-family model, and Qwen3 14B is a newer generation than Qwen2.5; their model generation and behavior may differ. Phi-4 is an exact match for the local Phi-4 model. These differences can change output quality, latency, and tokenization, so the API cost is an estimate based on local token counts, not a production bill forecast.

HF manages the serverless infrastructure. The table reports **Provider-managed; not disclosed** for machine count rather than estimating an actual fleet. The API total is token usage cost only.



In [ ]:
# Use the completed Part 1 records to build the two 90-row cost tables.
from collections import defaultdict
from html import escape
from math import ceil
from pathlib import Path
from statistics import mean, median
import json
from IPython.display import HTML, display

# Find the saved results without rerunning any models.
base = Path(globals().get("BASE_DIR", Path.cwd()))
candidates = [
    base / "benchmark_results.json",
    Path.cwd() / "docs/A7-part1/input/benchmark_results.json",
]
if globals().get("BENCHMARK_JSON_PATH"):
    candidates.insert(0, Path(BENCHMARK_JSON_PATH))
benchmark_path = next((path for path in candidates if path.is_file()), None)
if benchmark_path is None:
    raise FileNotFoundError("Place benchmark_results.json beside the notebook.")
with benchmark_path.open(encoding="utf-8") as file:
    records = json.load(file)["records"]

by_model = defaultdict(list)
for record in records:
    by_model[record["model_id"]].append(record)
if len(records) != 375 or len(by_model) != 15:
    raise ValueError(f"Expected 375 records from 15 models; found {len(records)} records.")

# Match each local model to a paid HF Inference Providers model with a public token price.
# Prices are USD per 1M input/output tokens, checked on 2026-10-26.
api_models = {
    "Qwen/Qwen3-0.6B": ("Qwen/Qwen3-4B-Instruct-2507", "Nscale", 0.01, 0.03,
                        "Same Qwen3 family; larger-model alternative (4B)"),
    "HuggingFaceTB/SmolLM2-360M-Instruct": ("ibm-granite/granite-4.2-3b", "DeepInfra", 0.03, 0.12,
                                            "Different model family; larger-model alternative (3B)"),
    "google/gemma-3-270m-it": ("google/gemma-3-4b-it", "DeepInfra", 0.05, 0.10,
                               "Same Gemma 3 family; larger-model alternative (4B)"),
    "meta-llama/Llama-3.2-1B-Instruct": ("meta-llama/Llama-3.1-8B-Instruct", "Novita", 0.02, 0.05,
                                         "Same Llama family; larger-model alternative (8B)"),
    "Qwen/Qwen3-1.7B": ("Qwen/Qwen3-4B-Instruct-2507", "Nscale", 0.01, 0.03,
                        "Same Qwen3 family; larger-model alternative (4B)"),
    "HuggingFaceTB/SmolLM2-1.7B-Instruct": ("ibm-granite/granite-4.2-3b", "DeepInfra", 0.03, 0.12,
                                            "Different model family; larger-model alternative (3B)"),
    "microsoft/Phi-4-mini-instruct": ("Qwen/Qwen3-4B-Instruct-2507", "Nscale", 0.01, 0.03,
                                      "Similar size and general instruction use; different model family"),
    "Qwen/Qwen3-4B": ("Qwen/Qwen3-4B-Instruct-2507", "Nscale", 0.01, 0.03,
                      "Same Qwen3 family and size; updated instruct variant"),
    "Qwen/Qwen2.5-7B-Instruct": ("Qwen/Qwen3.5-9B", "DeepInfra", 0.10, 0.15,
                                 "Related Qwen family and similar scale; newer model generation"),
    "mistralai/Mistral-7B-Instruct-v0.3": ("Qwen/Qwen3.5-9B", "DeepInfra", 0.10, 0.15,
                                           "Similar scale and general instruction use; different model family"),
    "Qwen/Qwen2.5-14B-Instruct": ("Qwen/Qwen3-14B", "Nscale", 0.07, 0.20,
                                  "Related Qwen family and similar size; newer model generation"),
    "microsoft/phi-4": ("microsoft/phi-4", "DeepInfra", 0.07, 0.14,
                        "Exact model match"),
    "mistralai/Mistral-Nemo-Instruct-2407": ("Qwen/Qwen3-14B", "Nscale", 0.07, 0.20,
                                             "Similar scale and general instruction use; different model family"),
    "meta-llama/Llama-3.2-3B-Instruct": ("meta-llama/Llama-3.1-8B-Instruct", "Novita", 0.02, 0.05,
                                         "Same Llama family; larger-model alternative (8B)"),
    "meta-llama/Llama-3.1-8B-Instruct": ("meta-llama/Llama-3.1-8B-Instruct", "Novita", 0.02, 0.05,
                                         "Exact model match"),
}
if set(api_models) != set(by_model):
    missing = sorted(set(by_model) - set(api_models))
    extra = sorted(set(api_models) - set(by_model))
    raise ValueError(f"API mapping mismatch. Missing: {missing}; extra: {extra}.")
if len({details[0] for details in api_models.values()}) != 7:
    raise AssertionError("Expected seven distinct HF API models in the current mapping.")

scenarios = [
    ("Prototype", 1_000), ("Early Startup", 10_000),
    ("Growing Product", 100_000), ("Large Platform", 1_000_000),
    ("Mass Consumer App", 10_000_000), ("Global Platform", 100_000_000),
]
requests_per_dau = 5
aws_hourly_price = 1.204
utilization = 0.70
aws_rows, api_rows = [], []
aws_costs_by_dau, api_costs_by_dau = defaultdict(list), defaultdict(list)

for model_id, samples in by_model.items():
    category = samples[0]["model_category"]
    avg_input = mean(r["prompt_tokens"] for r in samples)
    avg_output = mean(r["output_tokens"] for r in samples)
    avg_latency = mean(r["latency_s"] for r in samples)
    typical_latency = median(r["latency_s"] for r in samples)
    avg_tps = mean(r["tokens_per_sec"] for r in samples)
    api_model, provider, input_rate, output_rate, fit = api_models[model_id]

    for tier, dau in scenarios:
        requests = dau * requests_per_dau
        input_tokens = round(requests * avg_input)
        output_tokens = round(requests * avg_output)
        total_tokens = input_tokens + output_tokens

        machines = max(1, ceil(requests * avg_latency / (86_400 * utilization)))
        aws_cost = machines * aws_hourly_price * 24
        aws_costs_by_dau[dau].append(aws_cost)
        aws_rows.append([
            model_id, category, f"{tier} ({dau:,})", f"{requests:,}",
            f"{avg_input:.2f} / {avg_output:.2f}", f"{avg_latency:.3f}",
            f"{typical_latency:.3f}", f"{avg_tps:.2f}",
            f"{input_tokens:,}", f"{output_tokens:,}", f"{total_tokens:,}",
            f"{machines:,}", f"${aws_hourly_price:.3f}", f"${aws_cost:,.2f}",
        ])

        input_cost = input_tokens * input_rate / 1_000_000
        output_cost = output_tokens * output_rate / 1_000_000
        api_cost = input_cost + output_cost
        api_costs_by_dau[dau].append(api_cost)
        api_rows.append([
            category, model_id, api_model, provider, fit,
            f"${input_rate:.2f}", f"${output_rate:.2f}",
            f"{tier} ({dau:,})", f"{requests:,}",
            f"{input_tokens:,}", f"{output_tokens:,}", f"{total_tokens:,}",
            f"${input_cost:,.6f}", f"${output_cost:,.6f}", f"${api_cost:,.6f}",
            "Provider-managed; not disclosed", f"${api_cost:,.6f}",
        ])

if len(aws_rows) != 90 or len(api_rows) != 90:
    raise AssertionError("Each table must contain 6 DAU tiers × 15 local models.")

def show_table(title, headers, rows):
    head = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
                   for row in rows)
    display(HTML(f"<h3>{escape(title)}</h3><div style='max-height:650px;overflow:auto'>"
                 f"<table style='border-collapse:collapse;font-size:12px'><thead><tr>{head}</tr></thead>"
                 f"<tbody>{body}</tbody></table></div>"
                 "<style>td,th{border:1px solid #bbb;padding:4px;text-align:left}</style>"))

show_table(
    "Step 2.1–2.3: AWS self-hosting (90 rows)",
    ["Local model", "Model size", "DAU tier", "Requests/day",
     "Mean input / output tokens per request", "Mean latency (s)", "Median latency (s)",
     "Output tokens/sec", "Daily input tokens", "Daily output tokens", "Daily total tokens",
     "AWS machines required", "Machine price/hour", "AWS total cost/day"],
    aws_rows,
)
show_table(
    "Step 2.4: HF Inference Providers API (90 rows)",
    ["Local model size", "Local model", "HF API model", "Provider", "Model match",
     "Input price ($/1M)", "Output price ($/1M)", "DAU tier", "Requests/day",
     "Daily input tokens", "Daily output tokens", "Daily total tokens",
     "Input API cost/day", "Output API cost/day", "Estimated API cost/day",
     "API infrastructure machines required", "Total API cost/day"],
    api_rows,
)

comparison = []
for tier, dau in scenarios:
    aws = aws_costs_by_dau[dau]
    api = api_costs_by_dau[dau]
    comparison.append([f"{tier} ({dau:,})",
                      f"${min(aws):,.2f}–${max(aws):,.2f}",
                      f"${min(api):,.2f}–${max(api):,.2f}"])
show_table(
    "Daily cost range across the 15 model alternatives",
    ["DAU tier", "AWS self-host cost/day", "HF API cost/day"],
    comparison,
)
print(f"Loaded {len(records)} benchmark responses for {len(by_model)} models from {benchmark_path}.")
print(f"Both tables have 90 rows and map to {len({details[0] for details in api_models.values()})} distinct HF API models.")
print("API prices are per 1M tokens; HF machine counts are not public.")
print("These estimates use average traffic and single-request benchmark data, not production load tests.")